# Model Training
This notebook demonstrates data loading, preprocessing, model training, comparison, and saving the best model.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

# 1. Load Data
applicants = pd.read_csv('../data/applicants.csv')
credit_history = pd.read_csv('../data/credit_history.csv')
loan_apps = pd.read_csv('../data/loan_applications.csv')
labels = pd.read_csv('../data/labels.csv')

# 2. Merge Data
df = applicants.merge(credit_history, on='customer_id')
df = df.merge(loan_apps, on='customer_id')
df = df.merge(labels, on='application_id')

# 3. Handle missing values
df['experience_years'] = df['experience_years'].fillna(df['experience_years'].median())
df['employment_type'] = df['employment_type'].fillna('Unknown')

# 4. Feature Selection
features = ['age', 'annual_income', 'credit_score', 'loan_amount', 'total_debt', 'active_loans', 'delayed_payments']
X = df[features]
y = df['risk_class']

# 5. Encoding and Scaling
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Model Training and Comparison
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

best_model = None
best_acc = 0

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f"{name} Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_model = model
        
print(f"\nBest Model: {best_model.__class__.__name__} with accuracy {best_acc:.4f}")

# 7. Save Models
import os
os.makedirs('../models', exist_ok=True)
with open('../models/credit_risk_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
    
print("Models saved successfully in ../models/")
